# Trajectory optimization for a 2 link planar manipulator


Import statements, check that casadi symbolic variables are working

In [1]:
import numpy as np
from casadi import *
x = MX.sym("x")
print(jacobian(sin(x),x))


cos(x)


Build a class for storing all manipulator properties and EoMs

In [1]:
## Class for Manipulator Dynamics
class Manipulator:

    def __init__(self, M1, M2, L1, L2, init_theta1, init_theta2, max_iter):

        # Manipulator parameters
        self.M1 = M1
        self.M2 = M2
        self.L1 = L1
        self.L2 = L2
        self.init_theta1 = init_theta1
        self.init_theta2 = init_theta2
        self.max_iter = max_iter

        # Define simulation parameters
        self.T = 30                                         # not sure what a reasonable amount is
        self.dt = 0.01
        self.n = self.T/self.dt
        self.g = 9.81

        #Initialize optimizer object
        self.opti = Opti()

        # Define arrays for system matrices
        # self.M = MX.sym('M', 4,4,self.n)      @Cathy, you set this as 4x4xn. Shouldn't it be 2x2xn
        self.M = MX.sym('M', 2,2,self.n)
        self.c = MX.sym('c', 2,1,self.n)
        self.G = MX.sym("G", 2,1,self.n)
        self.Xdot = MX.sym('Xdot', 4, self.n)
        self.eq_constraints = MX.sym('eq')

        # Define our decision variables
        self.X = MX.sym('X', 4, self.n)           # X = [theta1, theta2, theta1_dot, theta2_dot].T
        # self.tau = MX.sym('tau', self.n)          # tau = [tau1, tau2].T
        self.tau = MX.sym('tau', 2, self.n)         #modified to be length 2 along first dimension

    def M(self):
        # Sets up the array of mass matrices
        for k in range(0, self.n):
            [theta1, theta2] = [self.X[0, k], self.X[1, k]]
            self.M[1,1,k] = (self.M1 + self.M2) * self.L1**2
            self.M[1,2,k] = self.M2 * self.L1 * self.L2 * cos(theta1 - theta2)
            self.M[2,1,k] = self.M2 * self.L1 * self.L2 * cos(theta1 - theta2)
            self.M[2,2,k] = self.M2 * self.L2**2
        return self.M

    def c(self):
        for k in range(0, self.n):
            [theta1, theta2, theta1dot, theta2dot] = [self.X[0, k], self.X[1, k], self.X[2, k], self.X[3, k]]
            self.c[1,1,k] = self.M2*self.L1*self.L2*theta2dot**2 * sin(theta1-theta2)
            self.c[2,1,k] = -self.M2*self.L1*self.L2*theta1dot**2 * sin(theta1-theta2)
        return self.c

    def G(self):
        for k in range(0, self.n):
            [theta1, theta2] = [self.X[0, k], self.X[1, k]]
            self.G[1,1,k] = (self.M1+self.M2) * self.g * self.L1 * cos(theta1)
            self.G[2,1,k] = self.M2 *self.g *self.L2 * cos(theta2)
        return self.G

    def diff(self, k):
        # Returns the time derivative of X as defined in the doc, at timestep k
        for k in range(0, self.n):
            self.Xdot[0:2, k] = self.X[2:4, k]
            B = self.tau[:, k] - self.c[:, k] - self.G[:, k]
            M_inv = solve(self.M[:, :, k], MX.eye(2))
            self.Xdot[2:4, k] = M_inv @ B
        return self.Xdot

    def set_initial_states(self):
        # Define starting conditions
        #Example: self.opti.set_initial(T, 1)
        initial_X = [[self.init_theta1, self.init_theta2, 0, 0]]
        initial_X_MX = MX(initial_values)
        self.opti.set_initial(X, initial_X_MX)
        return

    def build_objectives(self):
        # Define objective function to minimize
        #K = 1/2 I * omega^2
        # self.opti.minimize(  (y-x**2)**2   )
        return

    def build_equality_constraints(self, X_start, X_finish, T, dt):
        # Build constraints as defined in the doc
        #Example: self.opti.subject_to(x+y>=1 ) #where x and y are opti variables
        x = 1
        return

    def solve_optimizer(self):
        p_opts = {"record_time":True}
        s_opts = {"max_iter": self.max_iter}
        self.opti.solver('ipopt', p_opts, s_opts)
        self.set_initial_states()
        self.build_objectives()
        self.build_equality_constraints()
        sol = opti.solve()
        return sol    

SyntaxError: incomplete input (4025248140.py, line 70)

In [4]:
# Just using this cell to check if casadi behaves the way i think it will
a = MX.sym('a',4,4)
b = MX.sym('b',4,1)
c = inv_minor(a)

RuntimeError: .../casadi/core/mx.cpp:1809: Not implemented

In [ ]:
#SX for scalar operations, MX for Matrix operations
# https://web.casadi.org/docs/#a-simple-test-problem For optimization stack

#direct shooting vs direct multiple shooting vs indirect vs all at once

NLP solvers available:
blocksqp
bonmin
ipopt
knitro
snopt
worhp
feasiblesqpmethod
qrsqp
scpgen
sqpmethod

QP solvers:
clp
cplex
fatrop
gurobi
highs
hpipm
hpmpc
ooqp
osqp
proxqp
qpoases
sqic
superscs
ipqp
nlpsol
qrqp

In [ ]:
opti = casadi.Opti()

x = opti.variable()
y = opti.variable()

opti.minimize(  (y-x**2)**2   )
opti.subject_to( x**2+y**2==1 )
opti.subject_to(       x+y>=1 )

opti.solver('ipopt')


sol = opti.solve()

print(sol.value(x))
print(sol.value(y))

# opti.set_initial(sol1.value_variables())
